# B2 — Contrastive Meta-Adaptation (CMA)

NT-Xent contrastive loss + meta-training. Pozitif çift = aynı pasajın iki dropout view'ı (ESimCSE).

**Bağımsız çalışır.** Tüm parametreler Hücre 2'de toplanmıştır — sadece o hücreyi düzenleyerek deneyi kontrol edebilirsin. Sonuçlar `./results/B2_CMA_*` altına kaydedilir.

In [ ]:
# ─── HÜCRE 1: Setup ────────────────────────────────────────────────────
# Colab tespit + pip install + Drive mount (opsiyonel) + arf_lib import
import os, sys, importlib
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    # Colab'da arf_lib.py'nin yerini bul — varsayım:
    # repo /content/CALMA/standalone_notebooks/ altında klonlu
    REPO_BASE = "/content/CALMA"
    NB_DIR = f"{REPO_BASE}/standalone_notebooks"
    if not os.path.exists(NB_DIR):
        !git clone https://github.com/hakanskn/CALMA.git {REPO_BASE} 2>&1 | tail -5
    sys.path.insert(0, NB_DIR)

    # Drive mount (sonuçları Drive'a yedeklemek istersen PARAMS'ta results_root değiştir)
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        print("Drive mount skipped:", e)

    !pip install -q transformers==4.41.0 datasets==2.19.0 tokenizers==0.19.0 accelerate==0.30.0 matplotlib seaborn
else:
    # Lokal: arf_lib.py notebook ile aynı klasörde
    NB_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
    sys.path.insert(0, NB_DIR)

import arf_lib
importlib.reload(arf_lib)
print(f"arf_lib loaded from: {arf_lib.__file__}")
print(f"GPU info: {arf_lib.gpu_info()}")

In [ ]:
# ─── HÜCRE 2: PARAMETRELER ─────────────────────────────────────────────
PARAMS = {
    "method_name":   "B2_CMA",
    "run_name":      "wt2_seed42",
    "seed":          42,
    "results_root":  "./results",

    "model_name":    "gpt2",
    "tokenizer_name": "gpt2",

    "dataset":       "wikitext2",
    "max_seq_len":   256,         # contrastive için kısa OK
    "batch_size":    16,
    "num_workers":   2,

    "learning_rate": 5e-5,
    "num_epochs":    3,
    "warmup_steps":  0,
    "grad_clip":     1.0,
    "weight_decay":  0.01,

    "limit_train_batches": 100,
    "limit_eval_batches":  50,

    "log_every_n_steps":        25,
    "save_checkpoints":         False,
    "checkpoint_every_n_steps": 500,
    "keep_last_n_checkpoints":  3,

    # ─ Method-specific (CMA)
    "cma_temperature":    0.07,
    "cma_inner_lr":       1e-4,
    "cma_inner_steps":    2,
    "cma_outer_lr":       5e-5,
    "cma_projection_dim": 128,
    "cma_hidden_size":    768,
}

In [ ]:
# ─── HÜCRE 3: Imports ──────────────────────────────────────────────────
import math, time, json
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from arf_lib import (
    seed_everything, get_device, gpu_info, gpu_peak_mb,
    GPT2Wrapper, BaseRouterAttention,
    StandaloneTrainer, load_text_dataloaders,
    run_pipeline, save_results, append_to_index, make_plots,
)

seed_everything(PARAMS["seed"])
print(f"Seed set: {PARAMS['seed']}")
print(f"Device: {get_device()}")

In [ ]:
# ─── HÜCRE 4: Method — Contrastive Meta-Adaptation ─────────────────────
class ContrastiveMetaAdaptation:
    requires_meta_loop = True
    owns_optimizer = True

    def __init__(self, model, params):
        self.model = model
        mp = params
        self.temperature = float(mp.get("cma_temperature", 0.07))
        self.inner_lr = float(mp.get("cma_inner_lr", 1e-4))
        self.inner_steps = int(mp.get("cma_inner_steps", 2))
        self.outer_lr = float(mp.get("cma_outer_lr", 5e-5))
        self.proj_dim = int(mp.get("cma_projection_dim", 128))
        hidden = int(mp.get("cma_hidden_size", 768))
        device = next(model.parameters()).device
        self.projector = nn.Sequential(
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, self.proj_dim),
        ).to(device)
        self.outer_optimizer = torch.optim.Adam(
            list(self.model.parameters()) + list(self.projector.parameters()),
            lr=self.outer_lr,
        )

    def set_mask_token_id(self, tid):
        pass

    def _views(self, batch):
        self.model.train()
        out1 = self.model(input_ids=batch["input_ids"], attention_mask=batch.get("attention_mask"), output_hidden_states=True)
        out2 = self.model(input_ids=batch["input_ids"], attention_mask=batch.get("attention_mask"), output_hidden_states=True)
        h1 = out1.hidden_states[-1].mean(1)
        h2 = out2.hidden_states[-1].mean(1)
        return F.normalize(self.projector(h1), dim=-1), F.normalize(self.projector(h2), dim=-1)

    def _nt_xent(self, z1, z2):
        bs = z1.size(0)
        z = torch.cat([z1, z2], dim=0)
        sim = (z @ z.T) / self.temperature
        mask = torch.eye(2*bs, dtype=torch.bool, device=z.device)
        sim = sim.masked_fill(mask, float("-inf"))
        labels = torch.cat([torch.arange(bs, device=z.device)+bs, torch.arange(bs, device=z.device)])
        return F.cross_entropy(sim, labels)

    def adapt(self, context):
        was = self.model.training
        inner_opt = torch.optim.SGD(
            list(self.model.parameters()) + list(self.projector.parameters()),
            lr=self.inner_lr,
        )
        for _ in range(self.inner_steps):
            z1, z2 = self._views(context)
            loss = self._nt_xent(z1, z2)
            inner_opt.zero_grad(); loss.backward(); inner_opt.step()
        if not was: self.model.eval()

    def meta_train_step(self, support, query):
        orig_m = {n: p.data.clone() for n, p in self.model.named_parameters()}
        orig_p = {n: p.data.clone() for n, p in self.projector.named_parameters()}
        self.adapt(support)
        z1, z2 = self._views(query)
        q_loss = self._nt_xent(z1, z2)
        self.outer_optimizer.zero_grad()
        q_loss.backward()
        for n, p in self.model.named_parameters():
            p.data.copy_(orig_m[n])
        for n, p in self.projector.named_parameters():
            p.data.copy_(orig_p[n])
        self.outer_optimizer.step()
        return float(q_loss.item())

print("ContrastiveMetaAdaptation tanımlandı.")

In [ ]:
def build_model_and_adapter(params):
    from transformers import GPT2LMHeadModel
    model = GPT2LMHeadModel.from_pretrained(params["model_name"])
    adapter = ContrastiveMetaAdaptation(model, params)
    print(f"CMA adapter → model {sum(p.numel() for p in model.parameters()):,} + proj {sum(p.numel() for p in adapter.projector.parameters()):,}")
    return model, adapter

In [ ]:
# ─── HÜCRE 6: Run ──────────────────────────────────────────────────────
# Tek satır: model + opsiyonel adapter inşa et, pipeline çalıştır
model, adapter = build_model_and_adapter(PARAMS)
run_dir, result = run_pipeline(model, PARAMS, adapter=adapter)
print("\n" + "=" * 60)
print(f"RUN DIR: {run_dir}")
print(f"Final test PPL: {result['final_metrics']['test_ppl']:.4f}")
print(f"Final test BPC: {result['final_metrics']['test_bpc']:.4f}")
print(f"Duration: {result['duration_seconds']:.1f}s")
print("=" * 60)

In [ ]:
# ─── HÜCRE 7: Display plots (inline) ───────────────────────────────────
from IPython.display import Image, display
import os
plots_dir = run_dir / "plots"
if plots_dir.exists():
    for p in sorted(plots_dir.glob("*.png")):
        print(f"📈 {p.name}")
        display(Image(filename=str(p)))
else:
    print("Plots not generated.")
print(f"\nAll outputs in: {run_dir}")
print(f"Index file: {Path(PARAMS['results_root']) / '_index.csv'}")